# Hands-on Lab: Final Project - Generative AI for Data Science
## Ford Car Price Prediction

### Project Objective
1. Clean and prepare used car dataset (duplicates, missing values, normalization).
2. Perform exploratory data analysis (fuel type analysis, transmission price outliers via boxplots).
3. Build and evaluate Linear, Polynomial, and Ridge regression models on single and multiple features.
4. Perform Grid Search on Ridge regression to optimize the regularization parameter (alpha).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
print('All libraries loaded successfully!')

In [2]:
# Load dataset
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0271EN-SkillsNetwork/labs/v1/m3/ford.csv'
try:
    df = pd.read_csv(url)
    print(f'Dataset shape: {df.shape}')
    print(df.head())
except Exception as e:
    print(f'Error loading dataset: {e}')

In [3]:
# Data Cleaning: drop duplicates & fill/drop missing values
df_clean = df.drop_duplicates().copy() if 'df' in locals() else pd.DataFrame()
if not df_clean.empty:
    df_clean = df_clean.dropna()
    print(f'Cleaned shape: {df_clean.shape}')

In [4]:
# EDA: Fuel type counts and Transmission vs Price outliers
if not df_clean.empty:
    print('Fuel type distribution:')
    print(df_clean['fuelType'].value_counts())
    
    # Boxplot for transmission price outliers
    plt.figure(figsize=(10, 5))
    sns.boxplot(x='transmission', y='price', data=df_clean)
    plt.title('Transmission vs Price (Outlier Analysis)')
    plt.show()

In [5]:
# Model Training & Grid Search for Ridge Hyperparameter
if not df_clean.empty:
    X = pd.get_dummies(df_clean[['year', 'mileage', 'tax', 'mpg', 'engineSize', 'transmission', 'fuelType']], drop_first=True)
    y = df_clean['price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Ridge Regression with Grid Search
    param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
    ridge_cv = GridSearchCV(Ridge(), param_grid, cv=5, scoring='r2')
    ridge_cv.fit(X_train, y_train)
    
    best_ridge = ridge_cv.best_estimator_
    y_pred = best_ridge.predict(X_test)
    print(f'Best alpha: {ridge_cv.best_params_}')
    print(f'Test R2 Score: {r2_score(y_test, y_pred):.4f}')
    print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')